In [1]:
import pandas as pd
import numpy as np
import time
from pathlib import Path
from collections import Counter

from gensim.models import Word2Vec, FastText

print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:
from pathlib import Path

ROOT = Path.cwd()

if not (ROOT / "data").exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "cleaned_tinystories.csv"
MODEL_DIR = ROOT / "models"

print("Project Root:", ROOT)
print("Dataset Path:", DATA_PATH)
print("Model Directory:", MODEL_DIR)

Project Root: E:\MYFILES\MyProjects\NLP\nlp-project-tinyGpt
Dataset Path: E:\MYFILES\MyProjects\NLP\nlp-project-tinyGpt\data\cleaned_tinystories.csv
Model Directory: E:\MYFILES\MyProjects\NLP\nlp-project-tinyGpt\models


In [3]:

df = pd.read_csv(DATA_PATH)

print("Dataset Shape:", df.shape)
print("\nAvailable Columns:")
print(df.columns.tolist())

df.head()


Dataset Shape: (15000, 5)

Available Columns:
['id', 'label', 'story_clean', 'text_processed', 'n_words']


,id,label,story_clean,text_processed,n_words
0,0,Conflict,"Once upon a time, there was a big whale swimmi...",once upon a time there be a big whale swimming...,152
1,1,Conflict,"Once upon a time, there was a bug named Benny....",once upon a time there be a bug name benny ben...,148
2,2,Dialogue,"Once upon a time, in a small house, there live...",once upon a time in a small house there live a...,152
3,3,Conflict,"Once upon a time, there was a little girl name...",once upon a time there be a little girl name l...,133
4,4,Dialogue,"Once upon a time, in a small country, there li...",once upon a time in a small country there live...,169


In [4]:

sentences = [
    text.split()
    for text in df["text_processed"].fillna("")
    if text.strip()
]

total_tokens = sum(len(sentence) for sentence in sentences)

print("Total Stories:", len(sentences))
print("Total Tokens:", f"{total_tokens:,}")
print("First Sentence:", sentences[0][:20])


Total Stories: 15000
Total Tokens: 2,602,648
First Sentence: ['once', 'upon', 'a', 'time', 'there', 'be', 'a', 'big', 'whale', 'swimming', 'in', 'the', 'ocean', 'the', 'whale', 'be', 'very', 'happy', 'because', 'the']


In [5]:

PARAMS = {
    "vector_size": 100,
    "window": 5,
    "min_count": 5,
    "epochs": 10,
    "workers": 4,
    "seed": 42
}

MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Hyperparameters:")
print(PARAMS)


Hyperparameters:
{'vector_size': 100, 'window': 5, 'min_count': 5, 'epochs': 10, 'workers': 4, 'seed': 42}


In [6]:

start_time = time.time()

cbow = Word2Vec(
    sentences=sentences,
    sg=0,
    **PARAMS
)

cbow_time = time.time() - start_time

cbow.save(str(MODEL_DIR / "cbow.model"))

print("CBOW Training Completed!")
print("Vocabulary Size:", len(cbow.wv))
print("Training Time:", round(cbow_time, 2), "seconds")


CBOW Training Completed!
Vocabulary Size: 3838
Training Time: 12.4 seconds


In [7]:

start_time = time.time()

skipgram = Word2Vec(
    sentences=sentences,
    sg=1,
    **PARAMS
)

skipgram_time = time.time() - start_time

skipgram.save(str(MODEL_DIR / "skipgram.model"))

print("Skip-gram Training Completed!")
print("Vocabulary Size:", len(skipgram.wv))
print("Training Time:", round(skipgram_time, 2), "seconds")


Skip-gram Training Completed!
Vocabulary Size: 3838
Training Time: 43.94 seconds


In [8]:

comparison = pd.DataFrame([
    {
        "Model": "CBOW",
        "Architecture": "Context to Target",
        "Vocabulary": len(cbow.wv),
        "Training Time": round(cbow_time, 2)
    },
    {
        "Model": "Skip-gram",
        "Architecture": "Target to Context",
        "Vocabulary": len(skipgram.wv),
        "Training Time": round(skipgram_time, 2)
    }
])

comparison


,Model,Architecture,Vocabulary,Training Time
0,CBOW,Context to Target,3838,12.40
1,Skip-gram,Target to Context,3838,43.94


In [9]:

test_words = ["king", "queen", "story", "happy", "boy"]

for word in test_words:
    print(f"\nWord: {word}")

    if word in cbow.wv:
        print("CBOW:", cbow.wv.most_similar(word, topn=5))
    else:
        print("CBOW: Word not in vocabulary")

    if word in skipgram.wv:
        print("Skip-gram:", skipgram.wv.most_similar(word, topn=5))
    else:
        print("Skip-gram: Word not in vocabulary")



Word: king
CBOW: [('queen', 0.7826416492462158), ('prince', 0.7133022546768188), ('kingdom', 0.6251165270805359), ('wizard', 0.6180423498153687), ('person', 0.6007879972457886)]
Skip-gram: [('queen', 0.7133632898330688), ('prince', 0.7090193629264832), ('kingdom', 0.6692796945571899), ('princess', 0.6047947406768799), ('knight', 0.6034743785858154)]

Word: queen
CBOW: [('king', 0.7826415300369263), ('prince', 0.6898084878921509), ('wizard', 0.6292781233787537), ('princess', 0.6175019145011902), ('witch', 0.6112520694732666)]
Skip-gram: [('king', 0.7133632302284241), ('prince', 0.6940166354179382), ('princess', 0.6899834871292114), ('crown', 0.6470743417739868), ('castle', 0.5680786967277527)]

Word: story
CBOW: [('value', 0.5356979966163635), ('tale', 0.5221514105796814), ('reminder', 0.4742748737335205), ('important', 0.4408424198627472), ('learn', 0.43251845240592957)]
Skip-gram: [('moral', 0.8295838236808777), ('considerate', 0.582532525062561), ('learning', 0.5553626418113708), ('

In [10]:

word_counts = Counter(
    word
    for sentence in sentences
    for word in sentence
)

rare_words = [
    word for word, count in word_counts.items()
    if count == 2 and len(word) > 6
][:10]

print("Rare Words:")
print(rare_words)


Rare Words:
['waffles', 'dishearten', 'brushing', 'pianist', 'background', 'mesmerised', 'fragrance', 'wetness', 'sweetly', 'entirely']


In [11]:

start_time = time.time()

ft = FastText(
    sentences=sentences,
    sg=1,
    min_n=3,
    max_n=6,
    bucket=200_000,
    **PARAMS
)

fasttext_time = time.time() - start_time

ft.save(str(MODEL_DIR / "fasttext.model"))

print("fastText Training Completed!")
print("Vocabulary Size:", len(ft.wv))
print("Training Time:", round(fasttext_time, 2), "seconds")


fastText Training Completed!
Vocabulary Size: 3838
Training Time: 113.1 seconds


In [12]:

test_words = ["pianist", "dinosaurr", "happpy", "puppys"]

for word in test_words:
    print(f"\nTesting Word: {word}")

    if word in cbow.wv:
        print("Word2Vec:", cbow.wv.most_similar(word, topn=3))
    else:
        print("Word2Vec: No vector available")

    print(
        "fastText:",
        ft.wv.most_similar(word, topn=3)
    )



Testing Word: pianist
Word2Vec: No vector available
fastText: [('piano', 0.7431046366691589), ('spanish', 0.6963887214660645), ('saxophone', 0.6781364679336548)]

Testing Word: dinosaurr
Word2Vec: No vector available
fastText: [('dinosaur', 0.9873911142349243), ('robot', 0.5717623829841614), ('cheetah', 0.5477985143661499)]

Testing Word: happpy
Word2Vec: No vector available
fastText: [('happy', 0.7772441506385803), ('joyful', 0.677651584148407), ('satisfied', 0.6514084339141846)]

Testing Word: puppys
Word2Vec: No vector available
fastText: [('puppy', 0.9725970029830933), ('puppet', 0.6990665793418884), ('pup', 0.6983839273452759)]


In [13]:

final_comparison = pd.DataFrame([
    {
        "Model": "CBOW",
        "Vocabulary": len(cbow.wv),
        "Training Time": round(cbow_time, 2)
    },
    {
        "Model": "Skip-gram",
        "Vocabulary": len(skipgram.wv),
        "Training Time": round(skipgram_time, 2)
    },
    {
        "Model": "fastText",
        "Vocabulary": len(ft.wv),
        "Training Time": round(fasttext_time, 2)
    }
])

final_comparison


,Model,Vocabulary,Training Time
0,CBOW,3838,12.40
1,Skip-gram,3838,43.94
2,fastText,3838,113.10


In [14]:

print("Saved Model Files:")

for file in MODEL_DIR.glob("*.model"):
    print(file.name, "-", round(file.stat().st_size / (1024 * 1024), 2), "MB")


Saved Model Files:
cbow.model - 3.04 MB
fasttext.model - 3.04 MB
skipgram.model - 3.04 MB



# Conclusion

In this assignment, three word embedding models were trained using the TinyStories dataset.

1. **CBOW:** Predicts a target word using its surrounding context. It is generally faster to train.
2. **Skip-gram:** Predicts surrounding context words using a target word. It can perform better on less frequent words.
3. **fastText:** Uses character-level n-grams to generate word representations. It can produce vectors for words that are not present in the training vocabulary.

All three models use 100-dimensional word vectors, a context window of 5, minimum word frequency of 5, and 10 training epochs.

The models were saved for future use and comparison.
